# DNABERT2 Full Fine-Tuning: Colab A100 Benchmark

This notebook is the Google Colab mirror of SeqTrainer's Alpine DNABERT2
full-fine-tuning benchmark. It uses the same implementation, configuration,
predefined CSV splits, seed, validation-only model selection, metrics, and
artifact contract.

**Required runtime:** NVIDIA A100. Select an A100 GPU before running the first
cell. A T4 fallback is deliberately excluded because changing BF16 or the batch
profile would create a different execution configuration.

The notebook reads the three shared split CSV files from Google Drive. Model
downloads and caches stay on Colab-local storage for speed. Checkpoints and
final outputs are written to Drive for persistence.


## Fixed scientific contract

| Setting | Value |
|---|---|
| Dataset | GSE144621 EP genomic-order CSV splits |
| Train/validation/test | Predefined and unchanged |
| Seed | 42 |
| Model | `zhihan1996/DNABERT-2-117M` |
| Model revision | `7bce263b15377fc15361f52cfab88f8b586abda0` |
| Training scope | Full encoder fine-tuning |
| Pooling | Mean |
| Precision | BF16 |
| Physical batch size | 4 |
| Gradient accumulation | 8 |
| Effective batch size | 32 |
| Maximum epochs | 4 |
| Learning rate | 3e-5 |
| Optimizer | AdamW |
| Weight decay | 0.01 |
| Warmup ratio | 0.1 |
| Gradient clipping | 1.0 |
| Early stopping | Validation MCC, patience 2 |
| Threshold | Selected on validation MCC only |
| Primary final metric | Held-out test MCC |
| Secondary final metric | Held-out test AUPRC |

Do not change these values for the canonical comparison with CNN-v2 and the
Alpine run.


## 1. Verify the Colab accelerator

In [ ]:
import subprocess

gpu_info = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    text=True,
).strip()
print("GPU:", gpu_info)

if "A100" not in gpu_info:
    raise RuntimeError(
        "This canonical benchmark requires a Colab A100. Change the runtime "
        "accelerator to A100 and rerun from the top."
    )


## 2. Set reproducible paths and versions

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
SEQTRAINER_COMMIT = "659cb28a59b44057607329babf16b196b69b1e6f"
REPO_DIR = Path("/content/SeqTrainer")

MINIFORGE_DIR = Path("/content/miniforge3")
ENV_DIR = Path("/content/envs/seqtrainer-dnabert2")
ENV_PYTHON = ENV_DIR / "bin" / "python"
ENV_SEQTRAINER = ENV_DIR / "bin" / "seqtrainer"

HF_HOME = Path("/content/huggingface")
PIP_CACHE_DIR = Path("/content/pip-cache")
LOCAL_DATA_DIR = REPO_DIR / "data" / "promoter_classification"

CONFIG_RELATIVE_PATH = Path(
    "notebooks/benchmarks_sg/dnabert_benchmark/"
    "dnabertalpine/config/dnabert2_finetune.toml"
)

print("Pinned SeqTrainer commit:", SEQTRAINER_COMMIT)
print("Python environment:", ENV_DIR)


## 3. Create the same Python 3.10 environment as Alpine

The notebook uses an isolated Miniforge environment so Colab's host Python
version cannot silently change the benchmark. Package versions match the Alpine
bundle. FlashAttention and Triton are removed because the canonical runner uses
the stable PyTorch attention path.


In [ ]:
import subprocess

installer = Path("/content/Miniforge3-Linux-x86_64.sh")
if not (MINIFORGE_DIR / "bin" / "conda").exists():
    subprocess.run(
        [
            "wget",
            "-q",
            "https://github.com/conda-forge/miniforge/releases/latest/download/"
            "Miniforge3-Linux-x86_64.sh",
            "-O",
            str(installer),
        ],
        check=True,
    )
    subprocess.run(
        ["bash", str(installer), "-b", "-p", str(MINIFORGE_DIR)],
        check=True,
    )

conda = MINIFORGE_DIR / "bin" / "conda"
if not ENV_PYTHON.exists():
    subprocess.run(
        [str(conda), "create", "-y", "-p", str(ENV_DIR), "python=3.10", "pip"],
        check=True,
    )

install_env = os.environ.copy()
install_env["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)

subprocess.run(
    [str(ENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
    env=install_env,
)
subprocess.run(
    [
        str(ENV_PYTHON),
        "-m",
        "pip",
        "install",
        "numpy==1.24.4",
        "pandas==2.0.3",
        "scikit-learn==1.3.2",
        "torch==2.2.2",
        "transformers==4.29.2",
        "einops==0.6.1",
        "packaging",
    ],
    check=True,
    env=install_env,
)
subprocess.run(
    [str(ENV_PYTHON), "-m", "pip", "uninstall", "-y", "triton", "flash-attn", "flash_attn"],
    check=False,
    env=install_env,
)


## 4. Check out the exact SeqTrainer implementation

In [ ]:
import shutil

if REPO_DIR.exists():
    current_commit = subprocess.run(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
    )
    if current_commit.returncode != 0 or current_commit.stdout.strip() != SEQTRAINER_COMMIT:
        shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    ["git", "-C", str(REPO_DIR), "checkout", "--detach", SEQTRAINER_COMMIT],
    check=True,
)
checked_out_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
assert checked_out_commit == SEQTRAINER_COMMIT

subprocess.run(
    [str(ENV_PYTHON), "-m", "pip", "install", "-e", str(REPO_DIR)],
    check=True,
    env=install_env,
)

CONFIG_PATH = REPO_DIR / CONFIG_RELATIVE_PATH
assert CONFIG_PATH.exists(), CONFIG_PATH
print("Repository:", REPO_DIR)
print("Commit:", checked_out_commit)
print("Config:", CONFIG_PATH)


## 5. Verify the pinned environment and BF16 support

In [ ]:
environment_check = r'''import json
import sys
import torch
import transformers
import pandas
import sklearn
import seqtrainer

payload = {
    "python": sys.version,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "pandas": pandas.__version__,
    "scikit_learn": sklearn.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cuda_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
        if torch.cuda.is_available() else None,
    "bf16_supported": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    "seqtrainer": seqtrainer.__file__,
}
print(json.dumps(payload, indent=2))
if not payload["cuda_available"]:
    raise SystemExit("CUDA is not available inside the pinned environment")
if "A100" not in (payload["cuda_device"] or ""):
    raise SystemExit("The pinned environment is not using an A100")
if not payload["bf16_supported"]:
    raise SystemExit("BF16 is not supported by this runtime")
'''

result = subprocess.run(
    [str(ENV_PYTHON), "-c", environment_check],
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("
===== STDOUT =====")
print(result.stdout)
print("
===== STDERR =====")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Pinned DNABERT2 A100 environment check failed. Read STDOUT/STDERR above."
    )


## 6. Mount Google Drive

The only optional manual path is `DRIVE_DATA_DIR`. Leave it as `None` to search
Drive for one directory containing all three required files. Set it to a full
Drive path only when multiple copies exist or automatic discovery is slow.


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

MY_DRIVE = Path("/content/drive/MyDrive")
SHARED_DRIVES = Path("/content/drive/Shareddrives")

# Optional example:
# DRIVE_DATA_DIR = Path("/content/drive/MyDrive/AIxBio/Promoter Classification/Data")
DRIVE_DATA_DIR = None

DRIVE_OUTPUT_DIR = (
    MY_DRIVE
    / "SeqTrainer"
    / "outputs"
    / "dnabert2_finetune_colab_seed42"
)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Persistent output directory:", DRIVE_OUTPUT_DIR)


## 7. Locate and stage the shared CSV splits

In [ ]:
SPLIT_FILES = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}


def contains_all_splits(directory):
    directory = Path(directory)
    return directory.is_dir() and all((directory / name).is_file() for name in SPLIT_FILES.values())


def discover_aixbio_data_directories():
    if DRIVE_DATA_DIR is not None:
        candidate = Path(DRIVE_DATA_DIR)
        if not contains_all_splits(candidate):
            raise FileNotFoundError(
                f"DRIVE_DATA_DIR does not contain all required splits: {candidate}"
            )
        return [candidate]

    # Search only AIxBio-like folders. Do not fall back to dnabertalpine copies,
    # because AIxBio is the retained source dataset for these Colab notebooks.
    aixbio_roots = []
    for child in MY_DRIVE.iterdir():
        if child.is_dir() and child.name.lower().replace(" ", "") in {"aixbio", "aixbio"}:
            aixbio_roots.append(child)
        elif child.is_dir() and child.name.lower().replace(" ", "") == "aixbio":
            aixbio_roots.append(child)

    # Also include the common explicit names in case the Drive mount exposes only one.
    for candidate in [MY_DRIVE / "AIxBio", MY_DRIVE / "AI x Bio"]:
        if candidate.exists() and candidate not in aixbio_roots:
            aixbio_roots.append(candidate)

    matches = []
    for root in aixbio_roots:
        for train_path in root.rglob(SPLIT_FILES["train"]):
            parent = train_path.parent
            if contains_all_splits(parent) and parent not in matches:
                matches.append(parent)

    return matches


data_directories = discover_aixbio_data_directories()
if not data_directories:
    raise FileNotFoundError(
        "No AIxBio Drive directory contains all three shared benchmark CSVs. "
        "Keep the retained source dataset under AIxBio/Promoter Classification/Data "
        "or set DRIVE_DATA_DIR explicitly."
    )

if len(data_directories) > 1:
    print("Multiple AIxBio dataset directories were found:")
    for index, path in enumerate(data_directories, start=1):
        print(f"{index}. {path}")
    raise RuntimeError(
        "Set DRIVE_DATA_DIR to the intended AIxBio directory from the options above, "
        "then rerun this cell."
    )

source_data_dir = data_directories[0]
print("Selected AIxBio source data directory:", source_data_dir)

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for split, filename in SPLIT_FILES.items():
    source = source_data_dir / filename
    target = LOCAL_DATA_DIR / filename
    shutil.copy2(source, target)
    print(f"{split}: {source} -> {target}")


## 8. Audit data identity, schema, and labels

In [ ]:
import hashlib
import json
import pandas as pd


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


dataset_audit = {}
for split, filename in SPLIT_FILES.items():
    path = LOCAL_DATA_DIR / filename
    frame = pd.read_csv(path)
    missing_columns = {"sequence", "label"}.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{path} is missing columns: {sorted(missing_columns)}")
    labels = sorted(frame["label"].dropna().unique().tolist())
    if labels != [0, 1]:
        raise ValueError(f"{split} labels must be [0, 1], found {labels}")
    dataset_audit[split] = {
        "path": str(path),
        "rows": int(len(frame)),
        "label_counts": {
            str(key): int(value)
            for key, value in frame["label"].value_counts().sort_index().items()
        },
        "sha256": sha256(path),
    }

print(json.dumps(dataset_audit, indent=2))
(DRIVE_OUTPUT_DIR / "input_split_audit.json").write_text(
    json.dumps(dataset_audit, indent=2),
    encoding="utf-8",
)


## 9. Assert that the Alpine configuration is unchanged

In [ ]:
import tomllib

with CONFIG_PATH.open("rb") as handle:
    benchmark_config = tomllib.load(handle)

assert benchmark_config["experiment"]["seed"] == 42
assert benchmark_config["split"]["seed"] == 42
assert benchmark_config["model"]["params"]["mode"] == "full_finetune"
assert benchmark_config["model"]["params"]["pooling"] == "mean"
assert benchmark_config["model"]["params"]["fine_tuning_scope"] == "all_encoder_layers"
assert benchmark_config["training"]["batch_size"] == 4
assert benchmark_config["training"]["max_epochs"] == 4
assert benchmark_config["training"]["learning_rate"] == 3e-5
assert benchmark_config["training"]["params"]["gradient_accumulation_steps"] == 8
assert benchmark_config["training"]["params"]["early_stopping_metric"] == "validation_mcc"
assert benchmark_config["evaluation"]["threshold_strategy"] == "validation_mcc"
assert benchmark_config["environment"]["precision"] == "bf16"

effective_batch_size = (
    benchmark_config["training"]["batch_size"]
    * benchmark_config["training"]["params"]["gradient_accumulation_steps"]
)
assert effective_batch_size == 32

print("Configuration checks passed")
print("Physical batch size:", benchmark_config["training"]["batch_size"])
print("Effective batch size:", effective_batch_size)
print("Maximum epochs:", benchmark_config["training"]["max_epochs"])
print("Learning rate:", benchmark_config["training"]["learning_rate"])
print("Precision:", benchmark_config["environment"]["precision"])


## 10. Configure authenticated model downloads when available

DNABERT2 is public, so an Hugging Face token is optional. Adding `HF_TOKEN` to
Colab Secrets increases download limits. The token is never printed or written
to the benchmark manifest.


In [ ]:
run_env = os.environ.copy()
run_env["HF_HOME"] = str(HF_HOME)
run_env["TRANSFORMERS_CACHE"] = str(HF_HOME)
run_env["TOKENIZERS_PARALLELISM"] = "false"
run_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
run_env["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    run_env["HF_TOKEN"] = hf_token
    run_env["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print("HF_TOKEN loaded from Colab Secrets")
else:
    print("No HF_TOKEN found; public unauthenticated download will be used")


## 11. Run the canonical full fine-tuning benchmark

This is the long-running cell. The best validation-MCC checkpoint and final
artifacts are written directly to Google Drive. Test labels are not used for
checkpoint or threshold selection.


In [ ]:
command = [
    str(ENV_SEQTRAINER),
    "benchmark",
    "run",
    str(CONFIG_PATH),
    "--base-dir",
    str(REPO_DIR),
    "--output-dir",
    str(DRIVE_OUTPUT_DIR),
    "--strict",
]

print("Running:")
print(" ".join(command))
subprocess.run(command, check=True, env=run_env)


## 12. Inspect all metrics and the held-out result

In [ ]:
metrics_path = DRIVE_OUTPUT_DIR / "metrics.csv"
history_path = DRIVE_OUTPUT_DIR / "history.csv"
manifest_path = DRIVE_OUTPUT_DIR / "manifest.json"
predictions_path = DRIVE_OUTPUT_DIR / "predictions.csv"

for required_path in [metrics_path, history_path, manifest_path, predictions_path]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

metrics = pd.read_csv(metrics_path)
display(metrics)

test_metrics = metrics.loc[metrics["split"] == "test"].iloc[0]
print("Final held-out test MCC:", round(float(test_metrics["mcc"]), 6))
print("Final held-out test AUPRC:", round(float(test_metrics["auprc"]), 6))
print("Validation-selected threshold:", round(float(test_metrics["threshold"]), 6))


## 13. Plot training diagnostics and final test curves

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, roc_curve

history = pd.read_csv(history_path)
display(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="train loss")
axes[0].plot(history["epoch"], history["validation_loss"], marker="o", label="validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history["epoch"], history["validation_mcc"], marker="o")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation MCC")
plt.tight_layout()
plt.show()

predictions = pd.read_csv(predictions_path)
test_predictions = predictions.loc[predictions["split"] == "test"].copy()
y_true = test_predictions["label"].to_numpy()
y_score = test_predictions["probability"].to_numpy()
y_pred = test_predictions["prediction"].to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=axes[0], colorbar=False)
axes[0].set_title("Held-out test confusion matrix")

fpr, tpr, _ = roc_curve(y_true, y_score)
axes[1].plot(fpr, tpr, label=f"AUROC={float(test_metrics['auroc']):.3f}")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="grey")
axes[1].set_xlabel("False-positive rate")
axes[1].set_ylabel("True-positive rate")
axes[1].legend()

precision, recall, _ = precision_recall_curve(y_true, y_score)
axes[2].plot(recall, precision, label=f"AUPRC={float(test_metrics['auprc']):.3f}")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].legend()
plt.tight_layout()
plt.show()


## 14. Verify the reproducibility artifacts

In [ ]:
required_artifacts = [
    "metrics.csv",
    "metrics.json",
    "predictions.csv",
    "manifest.json",
    "history.csv",
    "checkpoints/best_model.pt",
]

missing_artifacts = [
    relative_path
    for relative_path in required_artifacts
    if not (DRIVE_OUTPUT_DIR / relative_path).exists()
]
if missing_artifacts:
    raise FileNotFoundError(f"Missing benchmark artifacts: {missing_artifacts}")

print("Completed artifact set:")
for path in sorted(DRIVE_OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(DRIVE_OUTPUT_DIR), path.stat().st_size, "bytes")


## Interpretation

Use held-out **test MCC** as the primary comparison with CNN-v2 and test AUPRC
as the secondary comparison. The threshold shown in every metrics row was
selected using validation predictions only.

This notebook and the Alpine bundle share the same scientific and optimizer
configuration. Different GPU drivers or kernels can still cause small
floating-point differences; the manifest records the actual runtime
environment. Do not report an interrupted or hardware-modified run as the
canonical result.
